# Optimasi Parameter DS CV-ME Chart — per Kombinasi (δ × Parameter)
## Referensi: Nuriman et al. (2026) — JQMA
### Model: Y_ij = A + B*X_ij + e_ij
### Metrik: TARL (Time-Adjusted Run Length)
### Optimasi: GA dijalankan TERPISAH untuk setiap sel tabel
---
**VERSI MULTIPROCESSING** — menggunakan `multiprocessing.Pool` untuk
looping tingkat tinggi. Setiap sel tabel = 1 task independen di Pool.
GA berjalan serial di dalam setiap worker (tanpa nested parallelism).

## 1. Import Library

In [ ]:
import numpy as np
from scipy.stats import nct, ncf
from scipy.integrate import quad
import multiprocessing as mp
import warnings
import time
import itertools
import os

warnings.filterwarnings("ignore")
print(f"Library berhasil dimuat. CPU cores: {mp.cpu_count()}")

## 2. Parameter Global

In [ ]:
GAMMA_0      = 0.05   # CV in-control
N0           = 5      # target rata-rata ukuran sampel
H            = 500    # truncation TARL
DELTA_LIST   = [1.01, 1.05, 1.1, 1.2, 1.3, 1.5, 2.0]

N1_RANGE = [3, 4, 5]
N_MAX    = 31

DEFAULT_BOUNDS = [
    (1.0, 10.0),  # ku1
    (0.1,  5.0),  # kl1
    (0.1,  8.0),  # wu
    (0.1,  5.0),  # wl
    (0.5, 10.0),  # ku2
    (0.1,  5.0),  # kl2
]

POP_SIZE      = 40
N_GENERATIONS = 60
PATIENCE      = 15

## 3. Import Modul Optimasi
Semua fungsi komputasi ada di `optimasi_ds_cvme_multiprocessing.py`.

In [ ]:
from optimasi_ds_cvme_multiprocessing import (
    get_gamma_stars, gamma0_star_const, gamma_plus_star_const,
    gamma0_star_linear, gamma_plus_star_linear,
    mean_std_gamma_hat, cdf_gamma_hat, compute_limits,
    is_valid_limits, compute_P_ooc,
    tarl_from_beta, compute_TARL,
    buat_tabel_jurnal,
    cetak_tabel_jurnal, cetak_tabel_df,
    plot_perbandingan_tarl, simpan_semua,
)

print("Modul optimasi berhasil di-import.")

## 4. Verifikasi Model γ*

In [ ]:
print("Verifikasi gamma* (theta=0.05, eta=0.28, B=1, m=1, delta=1.3):")
g0s = gamma0_star_const(0.05, 0.05, 0.28, 1, 1)
gps = gamma_plus_star_const(1.3, 0.05, 0.05, 0.28, 1, 1)
print(f"  Model A: gamma*_0={g0s:.4f}, gamma*_+={gps:.4f}")

t_test = compute_TARL(1.3, 3, 25, 4.5178, 1.9070, 1.5126, 1.9069, 2.5046, 1.9068,
                      gamma0=0.05, theta=0.05, B=1, m=1, eta=0.28)
print(f"TARL(delta=1.3, H={H}, paper params): {t_test:.4f}")

## 5. Tabel 1 — Model A: Variasi θ | η=0.28, m=B=1

In [ ]:
theta_list = [0, 0.01, 0.03, 0.05]

hasil_A_theta = buat_tabel_jurnal(
    variasi_param=theta_list,
    nama_param="theta",
    base_params={"gamma0": GAMMA_0, "theta": 0.0,
                 "eta": 0.28, "B": 1, "m": 1},
    error_model="constant",
)

cetak_tabel_jurnal(hasil_A_theta, theta_list, "θ",
                   judul="Model A | η=0.28, m=B=1 | variasi θ")

## 6. Tabel 1 — Model A: Variasi η | θ=0, m=B=1

In [ ]:
eta_list = [0, 0.1, 0.3, 0.5]

hasil_A_eta = buat_tabel_jurnal(
    variasi_param=eta_list,
    nama_param="eta",
    base_params={"gamma0": GAMMA_0, "theta": 0.0,
                 "eta": 0.0, "B": 1, "m": 1},
    error_model="constant",
)

cetak_tabel_jurnal(hasil_A_eta, eta_list, "η",
                   judul="Model A | θ=0, m=B=1 | variasi η")

## 7. Tabel 1 — Model A: Variasi m | θ=0.05, η=0.28, B=1

In [ ]:
m_list = [1, 3, 5, 7]

hasil_A_m = buat_tabel_jurnal(
    variasi_param=m_list,
    nama_param="m",
    base_params={"gamma0": GAMMA_0, "theta": 0.05,
                 "eta": 0.28, "B": 1, "m": 1},
    error_model="constant",
)

cetak_tabel_jurnal(hasil_A_m, m_list, "m",
                   judul="Model A | θ=0.05, η=0.28, B=1 | variasi m")

## 8. Tabel 1 — Model A: Variasi B | θ=0.05, η=0.28, m=1

In [ ]:
B_list = [1, 2, 3, 4]

hasil_A_B = buat_tabel_jurnal(
    variasi_param=B_list,
    nama_param="B",
    base_params={"gamma0": GAMMA_0, "theta": 0.05,
                 "eta": 0.28, "B": 1, "m": 1},
    error_model="constant",
)

cetak_tabel_jurnal(hasil_A_B, B_list, "B",
                   judul="Model A | θ=0.05, η=0.28, m=1 | variasi B")

## 9. Tabel 3 — Model B: Variasi θ | ω=φ=0, m=B=1

In [ ]:
hasil_B_theta = buat_tabel_jurnal(
    variasi_param=theta_list,
    nama_param="theta",
    base_params={"gamma0": GAMMA_0, "theta": 0.0,
                 "omega": 0.0, "phi": 0.0, "B": 1, "m": 1},
    error_model="linear",
)

cetak_tabel_jurnal(hasil_B_theta, theta_list, "θ",
                   judul="Model B | ω=φ=0, m=B=1 | variasi θ")

## 10. Tabel 3 — Model B: Variasi ω | θ=φ=0, m=B=1

In [ ]:
omega_list = [0.0, 0.1, 0.2, 0.3]

hasil_B_omega = buat_tabel_jurnal(
    variasi_param=omega_list,
    nama_param="omega",
    base_params={"gamma0": GAMMA_0, "theta": 0.0,
                 "omega": 0.0, "phi": 0.0, "B": 1, "m": 1},
    error_model="linear",
)

cetak_tabel_jurnal(hasil_B_omega, omega_list, "ω",
                   judul="Model B | θ=φ=0, m=B=1 | variasi ω")

## 11. Tabel 3 — Model B: Variasi φ | θ=ω=0, m=B=1

In [ ]:
phi_list = [0.0, 0.1, 0.2, 0.3]

hasil_B_phi = buat_tabel_jurnal(
    variasi_param=phi_list,
    nama_param="phi",
    base_params={"gamma0": GAMMA_0, "theta": 0.0,
                 "omega": 0.0, "phi": 0.0, "B": 1, "m": 1},
    error_model="linear",
)

cetak_tabel_jurnal(hasil_B_phi, phi_list, "φ",
                   judul="Model B | θ=ω=0, m=B=1 | variasi φ")

## 12. Tabel 3 — Model B: Variasi m | θ=0.05, ω=φ=0.1, B=1

In [ ]:
hasil_B_m = buat_tabel_jurnal(
    variasi_param=m_list,
    nama_param="m",
    base_params={"gamma0": GAMMA_0, "theta": 0.05,
                 "omega": 0.1, "phi": 0.1, "B": 1, "m": 1},
    error_model="linear",
)

cetak_tabel_jurnal(hasil_B_m, m_list, "m",
                   judul="Model B | θ=0.05, ω=φ=0.1, B=1 | variasi m")

## 13. Tabel 3 — Model B: Variasi B | θ=0.05, ω=φ=0.1, m=1

In [ ]:
hasil_B_B = buat_tabel_jurnal(
    variasi_param=B_list,
    nama_param="B",
    base_params={"gamma0": GAMMA_0, "theta": 0.05,
                 "omega": 0.1, "phi": 0.1, "B": 1, "m": 1},
    error_model="linear",
)

cetak_tabel_jurnal(hasil_B_B, B_list, "B",
                   judul="Model B | θ=0.05, ω=φ=0.1, m=1 | variasi B")

## 14. Visualisasi Perbandingan TARL

In [ ]:
%matplotlib inline

plot_perbandingan_tarl(
    {
        "Model A — variasi θ": (hasil_A_theta, theta_list, "θ"),
        "Model B — variasi θ": (hasil_B_theta, theta_list, "θ"),
    },
    judul="Perbandingan TARL: Model A vs Model B",
)

## 15. Simpan Semua Hasil ke CSV

In [ ]:
print("Menyimpan semua hasil...")
simpan_semua({
    "tabel1_A_variasi_theta": (hasil_A_theta, theta_list, "theta"),
    "tabel1_A_variasi_eta":   (hasil_A_eta,   eta_list,   "eta"),
    "tabel1_A_variasi_m":     (hasil_A_m,     m_list,     "m"),
    "tabel1_A_variasi_B":     (hasil_A_B,     B_list,     "B"),
    "tabel3_B_variasi_theta": (hasil_B_theta, theta_list, "theta"),
    "tabel3_B_variasi_omega": (hasil_B_omega, omega_list, "omega"),
    "tabel3_B_variasi_phi":   (hasil_B_phi,   phi_list,   "phi"),
    "tabel3_B_variasi_m":     (hasil_B_m,     m_list,     "m"),
    "tabel3_B_variasi_B":     (hasil_B_B,     B_list,     "B"),
})
print("\nSelesai!")

## 16. DataFrame (opsional)

In [ ]:
df_A_theta = cetak_tabel_df(hasil_A_theta, theta_list, "theta")
if df_A_theta is not None:
    display(df_A_theta)